# Cadet ATR Project — Full Pipeline
**Synthetic-to-Real Domain Adaptation for IR Target Classification**

Run cells top to bottom. Each section maps to a phase in your semester plan.

---

## Cell 1 — Setup (run once per session)

In [ ]:
# Install all dependencies
!pip install -q torch torchvision timm kornia scikit-learn scikit-image \
             wandb diffusers accelerate grad-cam pandas seaborn tqdm

# Mount Google Drive — saves checkpoints across sessions
from google.colab import drive
drive.mount('/content/drive')

# Clone your project repo (replace with your GitHub URL)
# !git clone https://github.com/YOUR_USERNAME/cadet_atr.git
# %cd cadet_atr

# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')
print(f'PyTorch: {torch.__version__}')

## Cell 2 — Generate synthetic IR images (run once, ~2 hours)

In [ ]:
# Generates 200 images × 3 classes = 600 synthetic IR images
# Output: data/synthetic/aircraft/, data/synthetic/vessel/, data/synthetic/vehicle/
!python generate_synthetic.py --n 200 --classes aircraft vessel vehicle

## Cell 3 — Download and prepare real IR dataset

In [ ]:
# Option A: Kaggle FLIR dataset (recommended)
# !pip install -q kaggle
# !kaggle datasets download -d deepnewbie/flir-thermal-images-dataset
# !unzip -q flir-thermal-images-dataset.zip -d data/flir_raw/

# Option B: If you already have a real IR dataset, organise it like:
# data/real/
#   aircraft/  (thermal images of aircraft)
#   vessel/    (thermal images of ships/boats)
#   vehicle/   (FLIR car/truck images as stand-in for ground vehicles)

import os
for cls in ['aircraft', 'vessel', 'vehicle']:
    os.makedirs(f'data/real/{cls}', exist_ok=True)
print('Real data directories created.')
print('Now copy your real IR images into data/real/CLASS_NAME/')

## Cell 4 — Train baseline model (Weeks 5–6)

In [ ]:
import wandb
wandb.login()  # paste your API key from wandb.ai/settings

from utils.config import cfg
from models.convnext import build_model
from data.dataset import make_loaders
from training.trainer import Trainer

# Build ConvNeXt with ImageNet weights
model = build_model(model_name='convnext_tiny', num_classes=3)

# Build DataLoaders
train_loader, val_loader, synth_test_loader = make_loaders('synthetic')

# Train — logs to W&B automatically
trainer = Trainer(model, run_name='baseline', aug_level='baseline')
ckpt_path = trainer.fit(train_loader, val_loader)

print(f'Best checkpoint: {ckpt_path}')

## Cell 5 — Measure domain gap (Week 7) ← HEADLINE RESULT

In [ ]:
from data.dataset import make_real_loader
from evaluation.evaluator import measure_domain_gap
from utils.visualise import plot_tsne, plot_intensity_histograms

# Load best checkpoint
model.load_state_dict(torch.load(ckpt_path))

# Get real image loader
real_test_loader = make_real_loader()

# THE CORE EXPERIMENT — record these numbers in your results table
gap_results = measure_domain_gap(
    model, synth_test_loader, real_test_loader,
    save_path='results/confusion_baseline.png'
)

print(f"\nYour domain gap: {gap_results['domain_gap']:.1%}")
print("Record this as your baseline in the results table!")

In [ ]:
# t-SNE before adaptation
plot_tsne(
    model, synth_test_loader, real_test_loader,
    title='Feature space — BEFORE adaptation',
    save_path='results/tsne_before.png'
)

# Pixel intensity comparison
plot_intensity_histograms(
    synth_dir=cfg.synth_dir, real_dir=cfg.real_dir,
    save_path='results/intensity_histograms.png'
)

## Cell 6 — Strategy 1: Histogram matching (Week 8)

In [ ]:
from adaptation.strategies import build_reference_histogram, apply_histogram_matching

# Build reference distribution from real images
reference = build_reference_histogram(cfg.real_dir)

# Apply to all synthetic training images
apply_histogram_matching(
    synth_img_dir = cfg.synth_dir,
    output_dir    = 'data/synthetic_matched/',
    reference     = reference,
)

# Retrain on matched images, then re-measure gap
# (swap root to 'data/synthetic_matched/' in your DataLoader)

## Cell 7 — Strategy 2: Domain randomisation (Week 8)

In [ ]:
from models.convnext import build_model
from training.trainer import Trainer

# Retrain with 'extended' augmentation pipeline
model_dr  = build_model()
trainer_dr = Trainer(model_dr, run_name='domain_random', aug_level='extended')
ckpt_dr   = trainer_dr.fit(train_loader, val_loader)

model_dr.load_state_dict(torch.load(ckpt_dr))
gap_dr = measure_domain_gap(model_dr, synth_test_loader, real_test_loader)
print(f"Gap after domain randomisation: {gap_dr['domain_gap']:.1%}")

## Cell 8 — Strategy 3: Fine-tuning on real data (Weeks 9–10)

In [ ]:
from adaptation.strategies import RealDataFinetuner
from data.dataset import make_loaders

# Build real-image train/val loaders
real_train_loader, real_val_loader, _ = make_loaders('real')

# Fine-tune starting from domain-randomisation checkpoint
finetuner = RealDataFinetuner(
    model           = build_model(),
    checkpoint_path = ckpt_dr,
    save_path       = 'checkpoints/finetuned_best.pt',
)
adapted_model = finetuner.finetune(
    real_train_loader, real_val_loader, mode='head_only'
)

gap_ft = measure_domain_gap(adapted_model, synth_test_loader, real_test_loader)
print(f"Gap after fine-tuning: {gap_ft['domain_gap']:.1%}  ← your best result")

## Cell 9 — Final results figures (Week 12)

In [ ]:
from utils.visualise import plot_gap_reduction, plot_tsne

# t-SNE after adaptation
plot_tsne(
    adapted_model, synth_test_loader, real_test_loader,
    title='Feature space — AFTER adaptation',
    save_path='results/tsne_after.png'
)

# Bar chart — the main results figure for your report
plot_gap_reduction([
    {'name': 'Synthetic baseline',    'synth_acc': gap_results['synth_acc'], 'real_acc': gap_results['real_acc']},
    {'name': '+ Histogram matching',  'synth_acc': None, 'real_acc': None},  # fill in from your run
    {'name': '+ Domain randomisation','synth_acc': gap_dr['synth_acc'],      'real_acc': gap_dr['real_acc']},
    {'name': '+ Fine-tuning',         'synth_acc': gap_ft['synth_acc'],      'real_acc': gap_ft['real_acc']},
], save_path='results/gap_reduction.png')

print('All figures saved to results/')